In [1]:
from clocks.epigenetic_clocks import (Horvath, Hannum, PhenoAge, run_all_clocks,
                                      build_cpg_reference, clock_cpgs)
import pickle

dataset = {}
chron_ages = {}
female = {}          # ← now a dict, not a single Series

ALL_DATASETS = [
    # 27K
    #'GSE41037',
    #'GSE19711',
    # 450K
    'GSE40279',
    'GSE55763',
    'GSE111629',
    'GSE157131',
    'GSE42861',
    'GSE84727',
    # EPIC
    'GSE169156',
    'GSE210255',
    'GSE179325',
]

clocks = [
    Horvath(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Horvath/horvath.csv"),
    Hannum(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Hannum/hannum.csv", intercept=0.0),
    PhenoAge(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/PhenoAge/phenoage.csv"),
#    GrimAgeAdapter(                                                              
#        GrimAgeV2(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/GrimAge/GrimAgeV2.csv"),
#        chron_ages=chron_ages,
#        female=female,
#    ),
]

dataset, chron_ages = {}, {}
for data in ALL_DATASETS:
    with open(f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl', 'rb') as file:
        methylation = pickle.load(file)
    dataset[data]    = methylation['CpG']                      # raw, NaNs kept
    chron_ages[data] = methylation['meta'].set_index('ID')['age']

reference = build_cpg_reference(dataset, cpgs=clock_cpgs(clocks))

results = run_all_clocks(
    datasets         = dataset,
    save_path        = "/work/gr-fe/bryan/EpiHydra/results",
    clocks           = clocks,
    chron_ages       = chron_ages,
    impute_reference = reference,
)

Loaded 353 CpG coefficients + intercept.
Loaded 71 CpG coefficients.
Loaded 513 CpG coefficients + intercept.


/tmp/4963925/ipykernel_1602945/935599417.py:28: UserWarning: Hannum intercept is set to 0.0 (the default). This is almost certainly incorrect and will bias every prediction. Supply the correct value from the paper's supplementary materials.
  Hannum(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Hannum/hannum.csv", intercept=0.0),



  Clocks   : horvath_age, hannum_age, phenoage_age
  Datasets : GSE40279, GSE55763, GSE111629, GSE157131, GSE42861, GSE84727, GSE169156, GSE210255, GSE179325

  Dataset : GSE40279
  Samples : 656   CpGs : 473034
  Clock CpGs : 891 | partly missing: 0 (cohort median) | missing in all samples: 2 (0 from reference, 2 → clock fallback)

  → Running horvath_age (Horvath) …

  → Running hannum_age (Hannum) …

  → Running phenoage_age (PhenoAge) …

  Age prediction statistics:
       horvath_age  hannum_age  phenoage_age  chron_age
count       656.00      656.00        656.00     656.00
mean         61.69       68.78         56.15      64.04
std          12.78       13.11         13.30      14.74
min          16.08       20.05          6.52      19.00
max          95.22       99.68         99.42     101.00

  ✓ Saved → /work/gr-fe/bryan/EpiHydra/results/GSE40279/age_predictions.csv

  Dataset : GSE55763
  Samples : 2711   CpGs : 473864
  Clock CpGs : 891 | partly missing: 54 (cohort median) 

In [2]:
filepath = f'/work/gr-fe/bryan/EpiHydra/data/02_processed/GSE179325/methylation.pkl'

with open(filepath, 'rb') as file:
    methylation = pickle.load(file)
print(methylation['meta'].head())

           ID      Organism  platform    age gender  horvath_age  hannum_age  \
0  GSM5414278  Homo sapiens  GPL21145  103.0      F    52.173379   70.117698   
1  GSM5414279  Homo sapiens  GPL21145   35.0      F    14.860064   38.484917   
2  GSM5414280  Homo sapiens  GPL21145   75.0      F   107.039139   77.184284   
3  GSM5414281  Homo sapiens  GPL21145   59.0      F    36.768921   54.993993   
4  GSM5414282  Homo sapiens  GPL21145   82.0      F    54.691421   75.649605   

   phenoage_age  chron_age  
0     83.954863      103.0  
1     34.524291       35.0  
2    100.111486       75.0  
3     67.746204       59.0  
4     78.455736       82.0  


In [3]:
methylation['meta']['horvath_age'].describe()

count    574.000000
mean      42.055279
std       14.143587
min        5.199269
25%       33.319603
50%       44.185033
75%       51.975908
max      107.039139
Name: horvath_age, dtype: float64

In [5]:
import pandas as pd 

replace = True

for data in ALL_DATASETS :
    
    filepath = f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl'
    
    with open(filepath, 'rb') as file:
        methylation = pickle.load(file)
    print(methylation['meta'].head())

    dataset[data] = methylation['CpG']
    chron_ages[data] = methylation['meta'].set_index('ID')['age']

    # Only merge columns that don't already exist in meta
    if replace : 
        overlap = results[data].columns.intersection(methylation['meta'].columns)
        methylation['meta'] = methylation['meta'].drop(columns=overlap)
        
    new_cols = [col for col in results[data].columns if col not in methylation['meta'].columns]
    
    if new_cols:
        methylation['meta'] = pd.merge(
            methylation['meta'],
            results[data][new_cols],
            left_on='ID',
            right_index=True,
            how='left'
        )

    print(methylation['meta'].head())
    with open(filepath, 'wb') as file:
        pickle.dump(methylation, file)


          ID      Organism  platform   age gender       tissue  horvath_age  \
0  GSM989827  Homo sapiens  GPL13534  67.0      F  whole blood    52.161142   
1  GSM989828  Homo sapiens  GPL13534  89.0      F  whole blood    75.446216   
2  GSM989829  Homo sapiens  GPL13534  66.0      F  whole blood    62.602883   
3  GSM989830  Homo sapiens  GPL13534  64.0      F  whole blood    54.249671   
4  GSM989831  Homo sapiens  GPL13534  62.0      F  whole blood    62.947226   

   hannum_age  phenoage_age  chron_age  
0   67.134750     50.851113       67.0  
1   86.167865     72.484114       89.0  
2   68.953058     58.508157       66.0  
3   68.895559     59.575270       64.0  
4   67.932521     54.271417       62.0  
          ID      Organism  platform   age gender       tissue  horvath_age  \
0  GSM989827  Homo sapiens  GPL13534  67.0      F  whole blood    52.161142   
1  GSM989828  Homo sapiens  GPL13534  89.0      F  whole blood    75.446216   
2  GSM989829  Homo sapiens  GPL13534  66.0